[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C51_Data_Augmentation_Course/02_backtranslation/02_backtranslation.ipynb)

# 02 · 回译与释义（可控失真的模拟翻译器 + 保真-多样权衡前沿）

目标：用**参数可控的模拟翻译器**复现回译的全部性质——保语义的机制、多样性从哪来、
四类高危失效场景、以及「宽松生成 + 严格过滤」为什么优于「保守生成 + 不过滤」。

路线：模拟翻译器（distortion 可控）→ 回译管线 → 解码温度是多样性主源 →
四类高危场景与否定奇偶检查 → 三级过滤流水线 → **保真-多样权衡前沿与最优温度** →
与 EDA 的成本对比 → ✏️ 练习 → 📖 答案 → 🧪 有效成本胶囊。

> 心智模型：**语义层增强的可靠性不来自「生成得多好」，而来自「过滤得多严」。**

## 0 · 复用检验工具 + 一个可控失真的模拟翻译器

真实翻译模型不可用，但回译的**性质**完全可以用一个带 `distortion` 参数的模拟器复现——
而且参数可控让我们能做真实环境里做不到的实验（比如「把失真从 0 调到 0.5，看过滤器什么时候开始起作用」）。

In [ ]:
import numpy as np, math, collections, itertools
rng = np.random.default_rng(0)

POS_WORDS = {'好吃', '不错', '推荐', '干净', '很好', '满意', '喜欢'}
NEG_WORDS = {'难吃', '差', '脏', '失望', '糟糕'}
NEGATORS  = {'不', '没', '别'}
NEUTRAL   = ['这家', '店', '的', '菜', '服务', '环境', '价格', '味道', '朋友', '下次',
             '我们', '昨天', '一起', '去', '吃', '了', '感觉', '整体']
ENTITIES  = ['海底捞', '西湖', '张三']
NUMBERS   = ['3', '30', '100']

def rule_label(tokens, window=3):
    score = 0
    for i, t in enumerate(tokens):
        if t in POS_WORDS:
            neg = any(tokens[j] in NEGATORS for j in range(max(0, i-window), i))
            score += -1 if neg else 1
        elif t in NEG_WORDS:
            neg = any(tokens[j] in NEGATORS for j in range(max(0, i-window), i))
            score += 1 if neg else -1
    return 1 if score > 0 else 0

def make_sentence(r, length=10):
    toks = list(r.choice(NEUTRAL, size=length-2, replace=True))
    pos = int(r.integers(1, len(toks)))
    toks.insert(pos, str(r.choice(sorted(POS_WORDS if r.random() < 0.5 else NEG_WORDS))))
    if r.random() < 0.45:
        toks.insert(max(0, pos - int(r.integers(1, 3))), str(r.choice(sorted(NEGATORS))))
    if r.random() < 0.3:
        toks.insert(int(r.integers(0, len(toks))), str(r.choice(ENTITIES)))
    if r.random() < 0.3:
        toks.insert(int(r.integers(0, len(toks))), str(r.choice(NUMBERS)))
    return toks, rule_label(toks)

def fidelity(pairs):
    return 1.0 if not pairs else sum(1 for o, y, a in pairs if rule_label(a) == y)/len(pairs)

def ngrams(t, n):
    return [tuple(t[i:i+n]) for i in range(len(t)-n+1)]

def distinct_n(texts, n=2):
    tot, uniq = 0, set()
    for t in texts:
        g = ngrams(t, n); tot += len(g); uniq.update(g)
    return len(uniq)/tot if tot else 0.0

def jaccard(a, b):
    sa, sb = set(a), set(b)
    return len(sa & sb)/len(sa | sb) if (sa | sb) else 1.0

print('✅ 检验工具就绪')

In [ ]:
# ── 可控的模拟翻译器 ──
# 「翻译」= 把词映射到同义变体；temperature 控制选变体的随机性；
# distortion 控制「语义漂移」的概率（丢否定词 / 改情感强度 / 错译实体 / 改数字）
# ⚠️ 释义表必须**扩充词表**：每个词有自己独有的变体。
#    如果多个词映射到同一批变体（多对一），高温反而会**降低**词面多样性 ——
#    这是设计模拟器时踩到的一个真实陷阱，也提醒你真实释义模型也可能有同样的坍缩倾向。
PARAPHRASE = {
    '这家': ['本', '该'], '店': ['餐厅', '饭店'], '菜': ['菜品', '食物'],
    '服务': ['服务员', '接待'], '环境': ['氛围', '装修'], '价格': ['收费', '价位'],
    '味道': ['口味', '风味'], '感觉': ['觉得', '体会'], '整体': ['总体', '总的'],
    '好吃': ['美味', '可口'], '不错': ['优秀', '出色'], '很好': ['极好', '很棒'],
    '难吃': ['糟糕', '难以下咽'], '差': ['不行', '很糟'], '脏': ['不洁', '脏乱'],
    '干净': ['整洁', '清洁'], '推荐': ['值得去', '力荐'],
    '满意': ['满足', '称心'], '失望': ['遗憾', '扫兴'],
}
# 情感词的**所有释义变体也算情感词** —— 这样「合法释义」不改变规则标签，
# 标签破坏就只来自真正的语义漂移（丢否定词等），而不是词表不全的假象。
POS_WORDS = POS_WORDS | {v for w in list(POS_WORDS) for v in PARAPHRASE.get(w, [])}
NEG_WORDS = NEG_WORDS | {v for w in list(NEG_WORDS) for v in PARAPHRASE.get(w, [])}
STRENGTH_SHIFT = {'不错': '很好', '满意': '很好'}   # 强度漂移（危险，但两端都是正面词）

def translate(tokens, temperature, distortion, r):
    '''模拟一次翻译。temperature 控多样性；distortion 控语义漂移。'''
    out = []
    for t in tokens:
        # 语义漂移：以 distortion 概率触发四类高危失效
        if r.random() < distortion:
            kind = r.random()
            if t in NEGATORS and kind < 0.45:
                continue                                        # ① 丢否定词（最致命）
            if t in STRENGTH_SHIFT and kind < 0.7:
                out.append(STRENGTH_SHIFT[t]); continue          # ② 强度漂移
            if t in ENTITIES and kind < 0.85:
                out.append(str(r.choice([e for e in ENTITIES if e != t]))); continue  # ③ 实体错译
            if t in NUMBERS:
                out.append(str(r.choice([n for n in NUMBERS if n != t]))); continue   # ④ 数字改变
        # 正常释义：temperature 越高越可能换成变体
        if t in PARAPHRASE and r.random() < temperature:
            out.append(str(r.choice(PARAPHRASE[t])))
        else:
            out.append(t)
    return out if out else list(tokens[:1])

def back_translate(tokens, temperature, distortion, r, hops=1):
    '''回译 = 正向 + 反向（hops>1 表示多跳，误差会累积）。'''
    cur = list(tokens)
    for _ in range(hops):
        cur = translate(cur, temperature, distortion, r)   # 正向
        cur = translate(cur, temperature, distortion, r)   # 反向
    return cur

DATA = [make_sentence(np.random.default_rng(s)) for s in range(400)]
r = np.random.default_rng(7)
for t, y in DATA[:3]:
    bt = back_translate(t, 0.7, 0.05, r)
    print(f'原({"正" if y else "负"}): {" ".join(t)}')
    print(f'回译({"正" if rule_label(bt) else "负"}): {" ".join(bt)}\n')
print('✅ 模拟回译器就绪：temperature 控多样性、distortion 控语义漂移')

## 1 · 解码温度是多样性的主要来源

常见误解是「换中间语言就能增加多样性」。实际上**多样性主要来自反向翻译的解码策略**。
温度 0（贪心/beam）时所有副本几乎相同——这正是 Edunov et al. 2018 指出的问题。

In [ ]:
def make_copies(data, n_aug, temperature, distortion, seed=0, hops=1):
    r = np.random.default_rng(seed)
    pairs, aug = [], []
    for t, y in data:
        for _ in range(n_aug):
            b = back_translate(t, temperature, distortion, r, hops=hops)
            pairs.append((t, y, b)); aug.append((b, y))
    return pairs, aug

print(f"{'温度':>6s} {'保真度':>9s} {'distinct-2':>11s} {'副本间平均Jaccard':>18s}")
for temp in [0.0, 0.3, 0.7, 1.0]:
    pairs, aug = make_copies(DATA, 4, temp, 0.05, seed=11)
    # 同一原句的 4 个副本之间有多像？（越低越多样）
    sims = []
    for i in range(0, len(pairs), 4):
        grp = [pairs[i+k][2] for k in range(4)]
        sims += [jaccard(grp[a], grp[b]) for a in range(4) for b in range(a+1, 4)]
    print(f'{temp:>6.1f} {fidelity(pairs):>9.1%} {distinct_n([a for a,_ in aug],2):>11.4f} '
          f'{np.mean(sims):>18.3f}')

p0, a0 = make_copies(DATA, 4, 0.0, 0.05, seed=11)
p1, a1 = make_copies(DATA, 4, 1.0, 0.05, seed=11)
sims0 = []
for i in range(0, len(p0), 4):
    grp = [p0[i+k][2] for k in range(4)]
    sims0 += [jaccard(grp[a], grp[b]) for a in range(4) for b in range(a+1, 4)]
sims1 = []
for i in range(0, len(p1), 4):
    grp = [p1[i+k][2] for k in range(4)]
    sims1 += [jaccard(grp[a], grp[b]) for a in range(4) for b in range(a+1, 4)]
assert np.mean(sims0) > np.mean(sims1), '温度越高，副本之间越不像（越多样）'
assert np.mean(sims0) > 0.95, '温度 0 时 4 个副本几乎完全相同 —— 等于只增强了 1 份'
print(f'\n⚠️  温度 0 时副本间 Jaccard = {np.mean(sims0):.3f}（几乎相同）——')
print('   生成 4 个副本，实际只得到 1 份新数据，两次翻译的算力全浪费了。')
print('✅ 这就是 Edunov et al. 2018 的核心发现：**回译要用采样，不要用纯 beam**。')

### 多跳的误差累积：多样性涨了，保真度崩了

In [ ]:
print(f"{'跳数':>5s} {'保真度':>9s} {'distinct-2':>11s} {'与原文Jaccard':>14s}")
for hops in [1, 2, 3]:
    pairs, aug = make_copies(DATA, 2, 0.7, 0.08, seed=13, hops=hops)
    sim2orig = np.mean([jaccard(o, a) for o, _, a in pairs])
    print(f'{hops:>5d} {fidelity(pairs):>9.1%} {distinct_n([a for a,_ in aug],2):>11.4f} '
          f'{sim2orig:>14.3f}')

f1 = fidelity(make_copies(DATA, 2, 0.7, 0.08, seed=13, hops=1)[0])
f3 = fidelity(make_copies(DATA, 2, 0.7, 0.08, seed=13, hops=3)[0])
assert f3 < f1, '多跳会累积误差，保真度下降'
print(f'\n✅ 1 跳保真 {f1:.1%} -> 3 跳保真 {f3:.1%}。多跳是「用保真度换多样性」的坏交易 ——')
print('   因为同样的多样性可以用「提高温度 + 严格过滤」更便宜地拿到（见第 4 节）。')

## 2 · 四类高危场景与否定奇偶检查

回译产出的句子**总是通顺的**，即使语义已经漂移。「通顺性给了它虚假的可信度」。
一个粗糙但零成本的检查：**否定词计数的变化**。

In [ ]:
def negation_count(tokens):
    return sum(1 for t in tokens if t in NEGATORS)

def entity_set(tokens):
    return {t for t in tokens if t in ENTITIES}

def number_set(tokens):
    return {t for t in tokens if t in NUMBERS}

def key_component_check(orig, aug):
    '''判据 ①：关键成分一致性（零成本，挡致命错误）。'''
    return (negation_count(orig) == negation_count(aug)
            and entity_set(orig) == entity_set(aug)
            and number_set(orig) == number_set(aug))

pairs, _ = make_copies(DATA, 4, 0.7, 0.12, seed=17)     # 故意用较高 distortion
broken = [(o, y, a) for o, y, a in pairs if rule_label(a) != y]
print(f'总副本 {len(pairs)}, 标签被破坏 {len(broken)} ({len(broken)/len(pairs):.1%})')
print('\n被破坏的例子（注意句子仍然「通顺」）:')
for o, y, a in broken[:3]:
    print(f'  原({"正" if y else "负"}): {" ".join(o)}')
    print(f'  译({"正" if rule_label(a) else "负"}): {" ".join(a)}')
    print(f'    否定词数 {negation_count(o)} -> {negation_count(a)} | '
          f'关键成分一致? {key_component_check(o, a)}\n')

# 量化：关键成分检查挡住了多少破坏？
caught = sum(1 for o, y, a in broken if not key_component_check(o, a))
kept_ok = sum(1 for o, y, a in pairs if rule_label(a) == y and key_component_check(o, a))
print(f'关键成分检查抓住了 {caught}/{len(broken)} = {caught/len(broken):.1%} 的破坏样本')
assert caught / len(broken) > 0.5, '关键成分检查应能抓住多数破坏'
filtered = [(o, y, a) for o, y, a in pairs if key_component_check(o, a)]
print(f'过滤后: 保留 {len(filtered)}/{len(pairs)} ({len(filtered)/len(pairs):.1%}), '
      f'保真度 {fidelity(pairs):.1%} -> {fidelity(filtered):.1%}')
assert fidelity(filtered) > fidelity(pairs), '过滤应提升保真度'
print('\n✅ 一个几乎零成本的规则过滤器，把保真度从 '
      f'{fidelity(pairs):.1%} 提到 {fidelity(filtered):.1%}。')
print('   「粗糙但零成本的过滤器」往往比「精细但昂贵的修正器」更值得先做。')

## 3 · 三级过滤流水线：先便宜后贵

① 关键成分（零成本，挡致命）→ ② 相似度**双边**带（挡漂移 + 挡无效副本）→ ③ 模型一致性（可选）。

**判据 ② 是双边的**：相似度太低=语义漂了；太高=副本等于原文，白花了两次翻译。

In [ ]:
def similarity_band_check(orig, aug, lo=0.30, hi=0.85):
    '''判据 ②：双边过滤。太低=漂移；太高=没变化（无效副本）。'''
    s = jaccard(orig, aug)
    return lo <= s <= hi, s

def build_classifier(train_data, seed=0):
    VOCAB = sorted(set(NEUTRAL) | POS_WORDS | NEG_WORDS | NEGATORS | set(ENTITIES)
                   | set(NUMBERS) | {v for vs in PARAPHRASE.values() for v in vs}
                   | set(STRENGTH_SHIFT.values()))
    v2i = {w: i for i, w in enumerate(VOCAB)}
    def feat(t):
        x = np.zeros(len(VOCAB)+1)
        for w in t:
            if w in v2i: x[v2i[w]] += 1
        x[-1] = 1; return x
    X = np.stack([feat(t) for t, _ in train_data]); y = np.array([l for _, l in train_data])
    w = np.random.default_rng(seed).normal(size=X.shape[1])*0.01
    for _ in range(400):
        p = 1/(1+np.exp(-(X @ w)))
        w -= 0.3*(X.T @ (p-y)/len(y) + 1e-3*w)
    return (lambda t: int(feat(t) @ w > 0)), (lambda t: float(feat(t) @ w))

clf, score_fn = build_classifier(DATA)

def model_consistency_check(orig, aug, min_conf=0.5):
    '''判据 ③：两边预测相反且都较自信 -> 丢弃。'''
    so, sa = score_fn(orig), score_fn(aug)
    if (so > 0) == (sa > 0):
        return True
    return not (abs(so) > min_conf and abs(sa) > min_conf)

def pipeline(pairs, use_key=True, use_sim=True, use_model=False, lo=0.30, hi=0.85):
    kept, stage_drop = [], collections.Counter()
    for o, y, a in pairs:
        if use_key and not key_component_check(o, a):
            stage_drop['①关键成分'] += 1; continue
        if use_sim:
            ok, s = similarity_band_check(o, a, lo, hi)
            if not ok:
                stage_drop['②相似度太低' if s < lo else '②相似度太高'] += 1; continue
        if use_model and not model_consistency_check(o, a):
            stage_drop['③模型不一致'] += 1; continue
        kept.append((o, y, a))
    return kept, stage_drop

pairs, _ = make_copies(DATA, 4, 0.7, 0.10, seed=19)
print(f'过滤前: {len(pairs)} 个副本, 保真度 {fidelity(pairs):.1%}, '
      f'distinct-2 {distinct_n([a for _,_,a in pairs],2):.4f}\n')
for label, kw in [('仅 ①', dict(use_sim=False)),
                  ('① + ②', dict()),
                  ('① + ② + ③', dict(use_model=True))]:
    kept, drops = pipeline(pairs, **kw)
    print(f'{label:<12s} 保留 {len(kept):>4d} ({len(kept)/len(pairs):>5.1%}) | '
          f'保真度 {fidelity(kept):>6.1%} | distinct-2 {distinct_n([a for _,_,a in kept],2):.4f}')
    print(f'{"":12s} 各阶段丢弃: {dict(drops)}')

k1, _ = pipeline(pairs, use_sim=False)
k12, _ = pipeline(pairs)
k123, _ = pipeline(pairs, use_model=True)
assert fidelity(k12) >= fidelity(k1), '加相似度过滤应不降保真度'
assert fidelity(k123) >= fidelity(k12), '加模型一致性应进一步提升'
assert len(k123) <= len(k12) <= len(k1), '过滤越多保留越少'
print('\n✅ 「先便宜后贵」的多阶段过滤 —— 与 C43 模块 04 的大规模质量过滤是同一个模式。')

In [ ]:
# 双边过滤为什么必要：单边（只挡太低）会留下大量「等于原文」的无效副本
pairs_low_temp, _ = make_copies(DATA, 4, 0.15, 0.05, seed=23)
one_sided = [(o, y, a) for o, y, a in pairs_low_temp if jaccard(o, a) >= 0.30]
two_sided = [(o, y, a) for o, y, a in pairs_low_temp if 0.30 <= jaccard(o, a) <= 0.85]
identical = sum(1 for o, y, a in pairs_low_temp if o == a)
print(f'低温度(0.15)下 {len(pairs_low_temp)} 个副本里，与原文**完全相同**的有 {identical} 个')
print(f'单边过滤(只挡太低): 保留 {len(one_sided)}，其中无效副本仍在')
print(f'双边过滤:           保留 {len(two_sided)}')
assert len(two_sided) < len(one_sided), '双边过滤会额外丢掉「太像」的副本'
assert identical > 0, '低温度必然产生一批与原文完全相同的副本'
print('\n✅ 相似度太高的副本没有提供任何新信息，只是把数据复制了一遍 ——')
print('   还白花了两次翻译的算力。**双边过滤，不是单边。**')

## 4 · 保真-多样权衡前沿与最优温度

过滤把不同温度下的「过滤后保真度」拉到接近水平，代价是**保留率不同**。
于是温度选择变成一个纯成本问题：

$$\text{有效成本/可用副本} = \frac{2\times\text{生成成本}}{\text{保留率}}$$

In [ ]:
def sweep_temperature(temps, distortion=0.10, n_aug=4, seed=29):
    rows = []
    for temp in temps:
        pairs, _ = make_copies(DATA, n_aug, temp, distortion, seed=seed)
        kept, _ = pipeline(pairs)
        keep_rate = len(kept)/len(pairs)
        div = distinct_n([a for _, _, a in kept], 2) if kept else 0.0
        fid = fidelity(kept)
        eff_cost = (2.0 / keep_rate) if keep_rate > 0 else float('inf')
        rows.append((temp, fidelity(pairs), keep_rate, fid, div, eff_cost, keep_rate*div))
    return rows

print(f"{'温度':>6s} {'过滤前保真':>10s} {'保留率':>8s} {'过滤后保真':>10s} "
      f"{'过滤后多样':>10s} {'有效成本':>9s} {'保留×多样':>10s}")
rows = sweep_temperature([0.1, 0.3, 0.5, 0.7, 0.9, 1.1])
for t, f0, kr, f1_, d, ec, score in rows:
    print(f'{t:>6.1f} {f0:>10.1%} {kr:>8.1%} {f1_:>10.1%} {d:>10.4f} {ec:>9.1f} {score:>10.4f}')

fids_after = [r[3] for r in rows]
print(f'\n过滤后保真度的跨度: {min(fids_after):.1%} ~ {max(fids_after):.1%}'
      f'（比过滤前的 {min(r[1] for r in rows):.1%} ~ {max(r[1] for r in rows):.1%} 窄）')
best = max(rows, key=lambda r: r[6])
print(f'✅ 让「保留率 × 多样性」最大的温度 = {best[0]}（有效成本 {best[5]:.1f} 次生成/可用副本）')
assert best[0] > 0.1, '最优温度不应是最低的那个（那样多样性太低）'
print('\n**结论：有了过滤器之后，温度的作用从「保真 vs 多样」变成「要生成多少候选」**——')
print('  即纯成本问题。所以「宽松生成 + 严格过滤」优于「保守生成 + 不过滤」。')

In [ ]:
# 验证核心论断：宽松生成+严格过滤 vs 保守生成+不过滤
loose_pairs, _ = make_copies(DATA, 4, 0.9, 0.10, seed=31)
loose_kept, _ = pipeline(loose_pairs)
cons_pairs, _ = make_copies(DATA, 4, 0.2, 0.10, seed=31)

# ⚠️ 比较两组「多样性」时要小心指标的可比性：
#    distinct-n 依赖**集合大小**（集合越大、总 n-gram 越多、比值越低），
#    所以两个大小不同的集合直接比 distinct-n 是不公平的。
#    这里用一个**逐样本、与集合大小无关**的指标：
#        新意 novelty = 1 - Jaccard(原文, 增强)   —— 「这个副本相对原文变了多少」
def mean_novelty(prs):
    return float(np.mean([1 - jaccard(o, a) for o, _, a in prs])) if prs else 0.0

print(f'可用副本数: 宽松+过滤 {len(loose_kept)} vs 保守不过滤 {len(cons_pairs)}\n')
print(f"{'策略':<28s} {'保真度':>9s} {'平均新意':>9s} {'distinct-2':>11s}")
print(f'{"宽松生成(T=0.9)+严格过滤":<28s} {fidelity(loose_kept):>9.1%} '
      f'{mean_novelty(loose_kept):>9.3f} {distinct_n([a for _,_,a in loose_kept],2):>11.4f}')
print(f'{"保守生成(T=0.2)+不过滤":<28s} {fidelity(cons_pairs):>9.1%} '
      f'{mean_novelty(cons_pairs):>9.3f} {distinct_n([a for _,_,a in cons_pairs],2):>11.4f}')

assert fidelity(loose_kept) >= fidelity(cons_pairs) - 0.01, '过滤后的保真度不应更差'
assert mean_novelty(loose_kept) > mean_novelty(cons_pairs) * 1.5, \
    '宽松生成的副本相对原文变化明显更大（每个副本携带更多新信息）'
print('\n✅ 宽松生成 + 严格过滤：**保真度相当（由过滤保证），而每个副本的「新意」明显更高**。')
print(f'   新意 {mean_novelty(cons_pairs):.3f} -> {mean_novelty(loose_kept):.3f} '
      f'（{mean_novelty(loose_kept)/max(mean_novelty(cons_pairs),1e-9):.1f}×）')
print('   ⚠️ 注意 distinct-2 这一列**不能直接跨集合大小比较** —— 这也是模块 04 会展开的一个点：')
print('      多样性指标只适合「同尺寸对比」与「同方案的趋势监测」。')
print('   这条「宽松生成 + 严格过滤」的结论对所有生成式增强都成立（含模块 03 的 LLM 合成）。')

## 5 · 与 EDA 的成本对比：回译处在不太经济的中间地带

In [ ]:
# 相对算力成本（以 EDA = 1 为单位）
COST = {'EDA': 1, 'AEDA': 1, '回译(2次seq2seq)': 1000, 'LLM改写': 2000}

def eda_like(tokens, r, p=0.1):
    protect = NEGATORS | set(ENTITIES) | set(NUMBERS)
    out = [t for t in tokens if (t in protect) or (r.random() >= p)]
    return out if out else list(tokens[:1])

r = np.random.default_rng(37)
eda_pairs = [(t, y, eda_like(t, r)) for t, y in DATA for _ in range(4)]
bt_pairs, _ = make_copies(DATA, 4, 0.7, 0.10, seed=37)
bt_kept, _ = pipeline(bt_pairs)

print(f"{'方法':<22s} {'相对成本':>9s} {'保真度':>9s} {'多样性':>9s} {'成本/多样性':>12s}")
for name, cost, prs in [('EDA + 保护', COST['EDA'], eda_pairs),
                        ('回译 + 三级过滤', COST['回译(2次seq2seq)'], bt_kept)]:
    f = fidelity(prs); d = distinct_n([a for _, _, a in prs], 2)
    print(f'{name:<22s} {cost:>9d} {f:>9.1%} {d:>9.4f} {cost/max(d,1e-9):>12.0f}')

d_eda = distinct_n([a for _,_,a in eda_pairs], 2)
d_bt = distinct_n([a for _,_,a in bt_kept], 2)
print(f'\n回译的多样性是 EDA 的 {d_bt/d_eda:.2f} 倍，但成本是 {COST["回译(2次seq2seq)"]}倍。')
assert fidelity(bt_kept) > 0.95 and fidelity(eda_pairs) > 0.95, '两者过滤/保护后都应高保真'
print('\n✅ 诚实的结论：**回译处在一个不太经济的中间地带** ——')
print('   便宜要正则化 -> 用 EDA/AEDA；要真正的多样性与可控性 -> 直接上 LLM（模块 03）。')
print('   回译仍值得学，因为它的**方法论**（语义空间增强 + 严格过滤）对 LLM 改写完全适用。')

## ✏️ 练习 1：关键成分检查的通用版

实现 `component_check(orig, aug, checks)`：`checks` 是一个 `{名称: 提取函数}` 字典
（提取函数返回可比较的值，如计数或集合）。返回 `(是否全部一致, 不一致的名称列表)`。

In [ ]:
def component_check(orig, aug, checks):
    # TODO: 对每个 (name, fn)，比较 fn(orig) == fn(aug)；返回 (全一致?, 不一致名单)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
CHECKS = {'否定词数': negation_count, '实体集': entity_set, '数字集': number_set,
          '长度比': lambda t: len(t) // 5}
o = ['这家', '店', '不', '好吃', '海底捞', '3']
a1 = ['本', '餐厅', '不', '美味', '海底捞', '3']              # 全部一致
a2 = ['本', '餐厅', '美味', '海底捞', '3']                    # 丢了否定词
a3 = ['本', '餐厅', '不', '美味', '西湖', '3']                 # 实体错译
ok1, bad1 = component_check(o, a1, CHECKS)
assert ok1 and bad1 == [], (ok1, bad1)
ok2, bad2 = component_check(o, a2, CHECKS)
assert not ok2 and '否定词数' in bad2
ok3, bad3 = component_check(o, a3, CHECKS)
assert not ok3 and '实体集' in bad3
print(f'一致    : {ok1}, 不一致项 {bad1}')
print(f'丢否定词: {ok2}, 不一致项 {bad2}')
print(f'实体错译: {ok3}, 不一致项 {bad3}')
print('✅ 练习 1 通过：把「哪些成分必须保持」写成可配置的检查表 ——')
print('   换任务只需换 checks，管线不变。')

## ✏️ 练习 2：双边相似度带的最优区间

实现 `best_band(pairs, lo_candidates, hi_candidates, min_fidelity)`：
在所有 `(lo, hi)` 组合里，找**过滤后保真度 ≥ min_fidelity** 且
**保留率 × 多样性最大**的那一组。返回 `(lo, hi, 保留率, 保真度, 多样性)`；无可行解返回 `None`。

In [ ]:
def best_band(pairs, lo_candidates, hi_candidates, min_fidelity):
    # TODO: 枚举 (lo, hi)（要求 lo < hi）；先过 key_component_check，再过相似度带；
    #       计算 keep_rate / fidelity / distinct_n；在满足 min_fidelity 的组合里最大化 keep_rate*distinct
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pairs_t, _ = make_copies(DATA, 4, 0.8, 0.10, seed=41)
res = best_band(pairs_t, [0.1, 0.2, 0.3, 0.4], [0.7, 0.8, 0.9, 1.0], min_fidelity=0.97)
assert res is not None
lo, hi, kr, fid, div = res
print(f'最优带 [{lo}, {hi}]: 保留率 {kr:.1%}, 保真度 {fid:.1%}, 多样性 {div:.4f}')
assert lo < hi and fid >= 0.97
# 要求极高保真度时可能无解
assert best_band(pairs_t, [0.1], [1.0], min_fidelity=1.01) is None
# 更严的保真要求 -> 保留率不会更高
res_strict = best_band(pairs_t, [0.1, 0.2, 0.3, 0.4], [0.7, 0.8, 0.9, 1.0], 0.99)
if res_strict:
    assert res_strict[2] <= kr + 1e-9, '更严的保真要求通常保留率更低'
print('✅ 练习 2 通过：过滤带也是「约束下最优」——保真度是约束，保留率×多样性是目标')

## ✏️ 练习 3：有效成本

实现 `effective_cost(gen_cost_per_call, calls_per_copy, keep_rate)`：
返回**每个可用副本**的成本。`keep_rate = 0` 时返回 `float('inf')`。
再实现 `pick_temperature(rows)`：`rows` 是 `[(temp, keep_rate, diversity)]`，
返回让 `keep_rate * diversity` 最大的温度。

In [ ]:
def effective_cost(gen_cost_per_call, calls_per_copy, keep_rate):
    # TODO
    raise NotImplementedError

def pick_temperature(rows):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert effective_cost(0.001, 2, 0.5) == 0.004, '保留率 50% -> 每个可用副本要生成 2 组'
assert effective_cost(0.001, 2, 0.0) == float('inf')
assert effective_cost(0.001, 1, 1.0) == 0.001
rows_t = [(t, kr, d) for t, _, kr, _, d, _, _ in sweep_temperature([0.1, 0.5, 0.9])]
best_t = pick_temperature(rows_t)
assert best_t in [t for t, _, _ in rows_t]
print(f'候选: {[(round(t,1), round(kr,3), round(d,4)) for t, kr, d in rows_t]}')
print(f'最优温度: {best_t}')
print(f'该温度下每可用副本成本: '
      f'{effective_cost(0.001, 2, dict((t, kr) for t, kr, _ in rows_t)[best_t]):.5f}')
print('✅ 练习 3 通过：有了过滤器，温度选择就是纯成本优化')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def component_check(orig, aug, checks):
    bad = [name for name, fn in checks.items() if fn(orig) != fn(aug)]
    return (not bad), bad

In [ ]:
# 练习 2 参考答案
def best_band(pairs, lo_candidates, hi_candidates, min_fidelity):
    best = None
    for lo in lo_candidates:
        for hi in hi_candidates:
            if lo >= hi: continue
            kept = [(o, y, a) for o, y, a in pairs
                    if key_component_check(o, a) and lo <= jaccard(o, a) <= hi]
            if not kept: continue
            kr = len(kept)/len(pairs)
            fid = fidelity(kept)
            div = distinct_n([a for _, _, a in kept], 2)
            if fid >= min_fidelity:
                score = kr * div
                if best is None or score > best[0]:
                    best = (score, lo, hi, kr, fid, div)
    return None if best is None else (best[1], best[2], best[3], best[4], best[5])

In [ ]:
# 练习 3 参考答案
def effective_cost(gen_cost_per_call, calls_per_copy, keep_rate):
    if keep_rate <= 0: return float('inf')
    return gen_cost_per_call * calls_per_copy / keep_rate

def pick_temperature(rows):
    return max(rows, key=lambda r: r[1] * r[2])[0]

---
## 🧪 真实数据胶囊：回译 vs LLM 改写的成本对比

用公开的量级数字算清「回译到底贵在哪」，以及什么时候它仍然是对的选择。

In [ ]:
# 公开量级（可改成你自己的数字）
MT_MODEL_PARAMS = 300e6          # 一个 MarianMT/NLLB 量级的翻译模型
LLM_PARAMS = 8e9                 # 一个 8B 指令模型
AVG_TOKENS = 40                  # 平均句长
GPU_TFLOPS_EFF, GPU_USD_H = 150, 4.0

def local_gen_cost(params, n_tokens):
    '''自建生成成本：自回归逐 token，每 token 约 2*params FLOPs。'''
    flops = 2 * params * n_tokens
    return flops / (GPU_TFLOPS_EFF * 1e12) / 3600 * GPU_USD_H

bt_cost = 2 * local_gen_cost(MT_MODEL_PARAMS, AVG_TOKENS)          # 两次翻译
llm_local = local_gen_cost(LLM_PARAMS, AVG_TOKENS)
# API 计价：输入 $0.15/1M token、输出 $0.60/1M token（公开量级）
llm_api = (AVG_TOKENS * 2 / 1_000_000) * 0.15 + (AVG_TOKENS / 1_000_000) * 0.60

print(f"{'方法':<24s} {'每副本成本$':>13s} {'1000条×4副本 总成本$':>22s}")
for name, c in [('回译（自建 MT ×2）', bt_cost),
                ('LLM 改写（自建 8B）', llm_local),
                ('LLM 改写（API）', llm_api)]:
    print(f'{name:<24s} {c:>13.8f} {c*4000:>22.4f}')

assert bt_cost < llm_local, '同为自建时，300M 模型跑两次仍便宜于 8B 跑一次'
print(f'\n① 自建对比: 回译(2×300M) 比 LLM(1×8B) 便宜 {llm_local/bt_cost:.1f} 倍')
print(f'② 但 API 版 LLM 只要 ${llm_api*4000:.4f} 总成本 —— 绝对值极低，')
print('   而且不需要你部署与维护两个翻译模型（运维成本远超算力成本）。')
print('\n✅ 回译仍然是对的选择的三种情形：')
print('   · 必须完全离线（合规/内网），不能调 API')
print('   · 已有现成的高质量翻译模型与推理服务')
print('   · 需要处理的量极大（几百万条），API 费用累积起来才是主导项')

**🧪 胶囊练习**：实现 `augmentation_plan(n_samples, n_aug, keep_rate, cost_per_call, calls_per_copy)`：
返回 `{'calls':…, 'usable_copies':…, 'total_cost':…, 'cost_per_usable':…}`。
注意：要拿到 `n_samples*n_aug` 个**可用**副本，需要生成 `n_samples*n_aug/keep_rate` 组。

In [ ]:
def augmentation_plan(n_samples, n_aug, keep_rate, cost_per_call, calls_per_copy):
    # TODO: attempts = ceil(n_samples*n_aug / keep_rate)
    #       calls = attempts * calls_per_copy
    #       usable = n_samples*n_aug；total_cost = calls*cost_per_call
    raise NotImplementedError

In [ ]:
# 自测
plan = augmentation_plan(1000, 4, keep_rate=0.6, cost_per_call=bt_cost/2, calls_per_copy=2)
assert plan['usable_copies'] == 4000
assert plan['calls'] == math.ceil(4000/0.6) * 2
assert abs(plan['cost_per_usable'] - plan['total_cost']/4000) < 1e-12
print(f'要 4000 个可用副本（保留率 60%）:')
print(f'  需生成 {plan["calls"]:,} 次调用, 总成本 ${plan["total_cost"]:.4f}, '
      f'每可用副本 ${plan["cost_per_usable"]:.8f}')
# 保留率越低成本越高
plan_low = augmentation_plan(1000, 4, 0.2, bt_cost/2, 2)
assert plan_low['total_cost'] > plan['total_cost'] * 2
print(f'保留率降到 20%: 总成本涨到 ${plan_low["total_cost"]:.4f} '
      f'（{plan_low["total_cost"]/plan["total_cost"]:.1f}×）')
print('\n✅ 胶囊练习通过：**过滤器的严格程度直接乘进成本** ——')
print('   所以「宽松生成+严格过滤」要配合「保留率别太低」，第 4 节的最优温度就是这个平衡点。')

In [ ]:
# 📖 胶囊参考答案
def augmentation_plan(n_samples, n_aug, keep_rate, cost_per_call, calls_per_copy):
    usable = n_samples * n_aug
    attempts = math.ceil(usable / keep_rate)
    calls = attempts * calls_per_copy
    total = calls * cost_per_call
    return {'calls': calls, 'usable_copies': usable,
            'total_cost': total, 'cost_per_usable': total / usable}

### 小结
- 回译在**语义空间**而非词面空间操作，「保语义」是内建约束 → 保真度比 EDA 高一档。
- 但它的失效**更隐蔽**：产出句子总是通顺的，即使语义已漂移。**通顺性给了它虚假的可信度**。
- **多样性主要来自解码温度，不是中间语言**。温度 0 时 4 个副本几乎相同（Jaccard>0.95）——白花算力。
- **多跳会累积误差**，是「用保真换多样」的坏交易；同样多样性可用「高温 + 严格过滤」更便宜地拿到。
- **四类高危场景**：否定、程度、实体、数字。零成本的**关键成分检查**（否定词计数 + 实体集 + 数字集）能抓住多数破坏。
- **相似度过滤必须是双边的**：太低=漂移，太高=副本等于原文（无效且浪费）。
- **核心论断（对所有生成式增强都成立）：宽松生成 + 严格过滤 > 保守生成 + 不过滤。** 有了过滤器，温度就变成纯成本问题。
- 诚实的结论：**回译处在不太经济的中间地带**——便宜要正则化用 EDA/AEDA，要多样性与可控性直接上 LLM。学它是为了那套方法论。

下一站：**模块 03 · LLM 驱动的指令数据合成** —— 唯一能真正增加信息量的增强层次。